# 15 - Extract Exact Metadata from FITS Headers
 
This notebook loads all 95 CARMENES FITS files and extracts precise metadata from their headers:
 - BJD (time of observation)
 - RV and drift-corrected RV
 - FWHM and Contrast of the cross-correlation function (CCF)
 - Signal-to-noise ratio (SNR) for all spectral orders
 
All extracted metadata is stored in a clean `metadata.csv` file for future use.

In [3]:
import os
from astropy.io import fits
import pandas as pd
import numpy as np
from tqdm import tqdm

New path:

In [4]:
from pathlib import Path
Path("../outputs/exact_metadata/").mkdir(parents=True, exist_ok=True)

### Load file list

In [5]:
def load_file_list(path):
    with open(path, "r") as f:
        return ["../../aumicAE/" + line.strip().lstrip("../") for line in f if line.strip()]

vis_a_files = load_file_list("../../aumicAE/data/carvis_visA/vis_a_files.txt")
print(f"{len(vis_a_files)} VIS-A files found.")

95 VIS-A files found.


### Extract header metadata

In [6]:
def extract_metadata(fits_path):
    with fits.open(fits_path) as hdul:
        hdr = hdul[0].header
        meta = {}

        meta["filename"] = os.path.basename(fits_path)

        meta["BJD"] = hdr.get("HIERARCH CARACAL BJD", np.nan)
        meta["RV"] = hdr.get("HIERARCH CARACAL SERVAL RV", np.nan)
        meta["RV_CCF"] = hdr.get("HIERARCH CARACAL CCF RVC", np.nan)
        meta["FWHM"] = hdr.get("HIERARCH CARACAL CCF FWHM", np.nan)
        meta["CONTRAST"] = hdr.get("HIERARCH CARACAL CCF CONTRAST", np.nan)

        for i in range(61):
            meta[f"SNR_{i}"] = hdr.get(f"HIERARCH CARACAL FOX SNR {i}", np.nan)

        return meta

In [7]:
all_meta = [extract_metadata(f) for f in tqdm(vis_a_files)]
df_meta = pd.DataFrame(all_meta)

100%|██████████| 95/95 [00:00<00:00, 348.50it/s]


Save it:

In [8]:
df_meta.to_csv("../outputs/exact_metadata/metadata.csv", index=False)
print("Saved exact metadata to ../outputs/exact_metadata/metadata.csv")

Saved exact metadata to ../outputs/exact_metadata/metadata.csv
